In [0]:
-- 1) Install the extensions in the current database
CREATE EXTENSION IF NOT EXISTS lakebase_vector CASCADE;
CREATE EXTENSION IF NOT EXISTS lakebase_text;


-- 2) Example table
DROP TABLE IF EXISTS kb_docs;

CREATE TABLE kb_docs (
  id         BIGSERIAL PRIMARY KEY,
  title      TEXT NOT NULL,
  body       TEXT NOT NULL,
  embedding  public.VECTOR(3) NOT NULL,
  search_tsv TSVECTOR GENERATED ALWAYS AS (
    to_tsvector(
      'english',
      coalesce(title, '') || ' ' || coalesce(body, '')
    )
  ) STORED
);

-- 3) Load some sample rows
INSERT INTO kb_docs (title, body, embedding) VALUES
  (
    'Lakebase overview',
    'Lakebase is a managed Postgres database for applications and agents.',
    '[0.10, 0.20, 0.30]'
  ),
  (
    'Semantic retrieval',
    'Vector search finds semantically similar records even when wording differs.',
    '[0.18, 0.26, 0.34]'
  ),
  (
    'Keyword retrieval',
    'BM25 ranking improves exact-term recall for names, identifiers, and phrases.',
    '[0.80, 0.70, 0.60]'
  );

-- 4) Build the indexes
-- Cosine is the most common metric for text embeddings
CREATE INDEX kb_docs_embedding_ann
  ON kb_docs USING lakebase_ann (embedding public.vector_cosine_ops);

-- Important: build BM25 after data is present
CREATE INDEX kb_docs_search_bm25
  ON kb_docs USING lakebase_bm25 (search_tsv);

-- 5) Pure vector search
SELECT id, title, embedding
FROM kb_docs
ORDER BY embedding OPERATOR(public.<=>) '[0.18, 0.26, 0.34]'::public.vector(3)
LIMIT 5;

-- 6) Pure BM25 keyword search
-- BM25 is a smart keyword search.
-- It searches for the words you typed and then sorts the matching documents from best match to worst match.
-- Lower score = more relevant
SELECT
  d.id,
  d.title,
  d.search_tsv OPERATOR(public.<@>)
    public.to_bm25query(
      pg_catalog.to_tsvector(
        'english'::pg_catalog.regconfig,
        'agents postgres'
      ),
      'kb_docs_search_bm25'::pg_catalog.regclass
    ) AS bm25_score
FROM kb_docs AS d
ORDER BY bm25_score
LIMIT 5;